# ESPAC 2024 -> SQLite ETL

This notebook ingests `inputs/01_2_DD_DATOS_ABIERTOS_ESPAC_2024.zip` (nested ZIP structure), loads all BD CSV tables into SQLite, and stores the DD dictionary as metadata tables.


## 1) Setup

Run this cell first. It defines paths and imports.


In [1]:
from __future__ import annotations

import io
import re
import sqlite3
import zipfile
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'inputs').exists() and (PROJECT_DIR.parent / 'inputs').exists():
    PROJECT_DIR = PROJECT_DIR.parent
INPUT_ZIP = PROJECT_DIR / "inputs/01_2_DD_DATOS_ABIERTOS_ESPAC_2024.zip"
DB_PATH = PROJECT_DIR / "outputs/01_espac_2024.sqlite"

assert INPUT_ZIP.exists(), f"Missing input file: {INPUT_ZIP}"
print(f"Input ZIP: {INPUT_ZIP}")
print(f"SQLite DB: {DB_PATH}")


Input ZIP: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\inputs\01_2_DD_DATOS_ABIERTOS_ESPAC_2024.zip
SQLite DB: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\01_espac_2024.sqlite


## 2) Helper functions

These helpers read nested ZIP files, clean table/column names, parse numeric values, and parse DD dictionary files.


In [2]:
def clean_name(value: str) -> str:
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9_]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    if not value:
        value = "col"
    if value[0].isdigit():
        value = f"c_{value}"
    return value


def clean_columns(columns: Iterable[str]) -> list[str]:
    out = []
    seen = {}
    for col in columns:
        base = clean_name(str(col))
        n = seen.get(base, 0)
        seen[base] = n + 1
        out.append(base if n == 0 else f"{base}_{n+1}")
    return out


def parse_mixed_numeric(series: pd.Series) -> pd.Series:
    if series.dtype != "object":
        return series
    s = series.astype("string").str.strip()
    s = s.replace({"": pd.NA, "-": pd.NA, "NA": pd.NA, "N/A": pd.NA, "null": pd.NA})
    s_norm = s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    numeric = pd.to_numeric(s_norm, errors="coerce")
    success_rate = numeric.notna().mean() if len(numeric) else 0.0
    if success_rate >= 0.85:
        return numeric
    return s


def normalize_frame(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = clean_columns(df.columns)
    for c in df.columns:
        df[c] = parse_mixed_numeric(df[c])
    return df


def read_nested_zip_bytes(outer_zip: Path, inner_name_pattern: str) -> bytes:
    with zipfile.ZipFile(outer_zip) as z:
        names = z.namelist()
        matches = [n for n in names if re.search(inner_name_pattern, n, flags=re.I)]
        if not matches:
            raise FileNotFoundError(f"No inner zip matched pattern: {inner_name_pattern}")
        return z.read(matches[0])


def read_bd_tables(outer_zip: Path) -> dict[str, pd.DataFrame]:
    inner = read_nested_zip_bytes(outer_zip, r"INEC_ESPAC_BD_2024\.zip$")
    tables: dict[str, pd.DataFrame] = {}
    with zipfile.ZipFile(io.BytesIO(inner)) as z:
        for member in sorted(z.namelist()):
            if not member.lower().endswith(".csv"):
                continue
            raw = z.read(member)
            df = pd.read_csv(io.BytesIO(raw), sep=";", encoding="utf-8-sig", dtype=str, low_memory=False)
            table_name = clean_name(Path(member).stem.replace("_BD_2024", ""))
            tables[table_name] = normalize_frame(df)
    return tables


def parse_dd_file(raw_bytes: bytes, source_table: str) -> pd.DataFrame:
    try:
        tmp = pd.read_csv(io.BytesIO(raw_bytes), sep=";", header=None, dtype=str, encoding="utf-8", engine="python")
    except UnicodeDecodeError:
        tmp = pd.read_csv(io.BytesIO(raw_bytes), sep=";", header=None, dtype=str, encoding="latin-1", engine="python")
    tmp = tmp.fillna("")

    marker = "CODIGO DE LA VARIABLE"
    header_idx = None
    marker_col = None
    for i in range(len(tmp)):
        row = tmp.iloc[i].astype(str).str.strip()
        matches = row.str.upper() == marker
        if matches.any():
            header_idx = i
            marker_col = int(np.where(matches.values)[0][0])
            break

    cols = ["source_table", "codigo_variable", "nombre_variable", "pregunta", "tipo_variable", "formato_dato"]
    if header_idx is None or marker_col is None:
        return pd.DataFrame(columns=cols)

    data = tmp.iloc[header_idx + 1 :, marker_col : marker_col + 5].copy()
    data.columns = cols[1:]
    data = data.apply(lambda c: c.astype(str).str.strip())
    code = data["codigo_variable"].astype(str).str.strip()
    valid = code.str.match(r"^[A-Za-z][A-Za-z0-9_]*$", na=False)
    banned = code.str.upper().isin({"CODIGO", "NOMBRE", "PREGUNTA", "TIPO", "FORMATO", "FUENTE", "ELABORADO"})
    data = data[(code != "") & (code.str.lower() != "nan") & valid & (~banned)]
    data.insert(0, "source_table", source_table)
    return data.reset_index(drop=True)


def read_dd_dictionary(outer_zip: Path) -> pd.DataFrame:
    inner = read_nested_zip_bytes(outer_zip, r"INEC_ESPAC_DD_2024\.zip$")
    rows = []
    with zipfile.ZipFile(io.BytesIO(inner)) as z:
        for member in sorted(z.namelist()):
            if not member.lower().endswith(".csv"):
                continue
            source_table = clean_name(Path(member).stem.replace("_DD_2024", "").replace("INEC_", ""))
            rows.append(parse_dd_file(z.read(member), source_table))
    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)


## 3) Load BD tables and DD dictionary


In [3]:
bd_tables = read_bd_tables(INPUT_ZIP)
print(f"Loaded {len(bd_tables)} BD tables")
for t, df in bd_tables.items():
    print(f"- {t:12s} rows={len(df):>7,d} cols={len(df.columns):>3d}")

dd_dictionary = read_dd_dictionary(INPUT_ZIP)
print(f"\nDictionary rows: {len(dd_dictionary):,}")
dd_dictionary.head()


Loaded 19 BD tables
- inec_acnac   rows= 40,887 cols= 41
- inec_adnac   rows=  7,180 cols= 24
- inec_apnac   rows= 40,869 cols= 83
- inec_cgnac   rows= 40,887 cols= 23
- inec_cpnac   rows= 12,747 cols=110
- inec_ctnac   rows= 15,821 cols=115
- inec_eunac   rows= 27,862 cols= 24
- inec_fpnac   rows=    612 cols= 74
- inec_ftnac   rows=    317 cols= 76
- inec_glnac   rows= 12,515 cols= 89
- inec_gpnac   rows=  4,599 cols= 34
- inec_gvnac   rows=  2,171 cols= 19
- inec_oenac   rows=  5,974 cols= 16
- inec_pcnac   rows= 19,522 cols= 75
- inec_sunac   rows= 90,247 cols= 29
- inec_vgonac  rows=  2,351 cols= 31
- inec_vgpnac  rows=  4,595 cols= 31
- inec_vgvnac  rows= 12,509 cols= 44
- inec_vmnac   rows= 56,760 cols= 23

Dictionary rows: 906


,source_table,codigo_variable,nombre_variable,pregunta,tipo_variable,formato_dato
0,adnac,Identificador,Identificador,*,Variable identificadora,Numérico
1,adnac,ual_prov,Provincia,*,Variable identificadora,Numérico
2,adnac,ual_estr,Estrato,*,Variable identificadora,Numérico
3,adnac,ual_segm,Segmento,*,Variable identificadora,Numérico
4,adnac,al_ncues,Cuestionario,*,Variable identificadora,Numérico


## 4) Build core relational tables

- `encuestas`: one row per `identificador`
- One table per BD file
- `dd_variables`: variable dictionary
- `table_inventory`: table summary


In [4]:
def build_encuestas(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    pieces = []
    key_cols = ["identificador", "ual_prov", "ual_estr", "ual_segm", "al_ncues"]
    for df in tables.values():
        available = [c for c in key_cols if c in df.columns]
        if "identificador" not in available:
            continue
        chunk = df[available].copy().rename(columns={
            "ual_prov": "provincia",
            "ual_estr": "estrato",
            "ual_segm": "segmento",
            "al_ncues": "cuestionario",
        })
        pieces.append(chunk)

    if not pieces:
        return pd.DataFrame(columns=["identificador", "provincia", "estrato", "segmento", "cuestionario"])

    all_ids = pd.concat(pieces, ignore_index=True)
    all_ids = all_ids.sort_values(by=[c for c in all_ids.columns if c != "identificador"])
    return all_ids.drop_duplicates(subset=["identificador"], keep="first").reset_index(drop=True)


encuestas = build_encuestas(bd_tables)
print(f"encuestas rows: {len(encuestas):,}")
encuestas.head()


encuestas rows: 40,887


,identificador,provincia,estrato,segmento,cuestionario
0,01135101000330001,AZUAY,01,00033,0001
1,01135101000339999,AZUAY,01,00033,9999
2,01015701001030001,AZUAY,01,00103,0001
3,01135101001690001,AZUAY,01,00169,0001
4,01135101001699999,AZUAY,01,00169,9999


In [5]:
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

encuestas.to_sql("encuestas", conn, if_exists="replace", index=False)
conn.execute("CREATE UNIQUE INDEX idx_encuestas_identificador ON encuestas(identificador);")

inventory = []
for table_name, df in bd_tables.items():
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    inventory.append({
        "table_name": table_name,
        "row_count": len(df),
        "column_count": len(df.columns),
        "has_identificador": int("identificador" in df.columns),
    })
    if "identificador" in df.columns:
        conn.execute(f"CREATE INDEX idx_{table_name}_identificador ON {table_name}(identificador);")

dd_dictionary.to_sql("dd_variables", conn, if_exists="replace", index=False)
pd.DataFrame(inventory).to_sql("table_inventory", conn, if_exists="replace", index=False)

conn.commit()
conn.close()
print(f"SQLite database created: {DB_PATH}")


SQLite database created: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\01_espac_2024.sqlite


## 5) Add explicit foreign-key constrained copies (optional)

SQLite cannot add FK constraints to existing tables directly, so this step creates constrained copies (`rel_*`) for tables with `identificador`.


In [6]:
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

with_ident = pd.read_sql_query("SELECT table_name FROM table_inventory WHERE has_identificador = 1", conn)["table_name"].tolist()
created = []

for t in with_ident:
    rel_t = f"rel_{t}"
    cols = pd.read_sql_query(f"PRAGMA table_info({t})", conn)
    col_defs = []
    for _, r in cols.iterrows():
        name = r["name"]
        ctype = "TEXT" if name == "identificador" else "NUMERIC"
        col_defs.append(f'"{name}" {ctype}')

    ddl = (
        f"DROP TABLE IF EXISTS {rel_t};\n"
        f"CREATE TABLE {rel_t} (" + ", ".join(col_defs) + ", FOREIGN KEY(identificador) REFERENCES encuestas(identificador));\n"
        f"INSERT INTO {rel_t} SELECT * FROM {t};\n"
        f"CREATE INDEX IF NOT EXISTS idx_{rel_t}_identificador ON {rel_t}(identificador);"
    )
    conn.executescript(ddl)
    created.append(rel_t)

conn.commit()
conn.close()
print(f"Created {len(created)} constrained relational copies.")
created[:10]


Created 19 constrained relational copies.


['rel_inec_acnac',
 'rel_inec_adnac',
 'rel_inec_apnac',
 'rel_inec_cgnac',
 'rel_inec_cpnac',
 'rel_inec_ctnac',
 'rel_inec_eunac',
 'rel_inec_fpnac',
 'rel_inec_ftnac',
 'rel_inec_glnac']

## 6) Validate outputs


In [7]:
conn = sqlite3.connect(DB_PATH)
print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn))
print(pd.read_sql_query("SELECT * FROM table_inventory ORDER BY row_count DESC LIMIT 10", conn))
print(pd.read_sql_query("SELECT * FROM dd_variables LIMIT 10", conn))
conn.close()


               name
0      dd_variables
1         encuestas
2        inec_acnac
3        inec_adnac
4        inec_apnac
5        inec_cgnac
6        inec_cpnac
7        inec_ctnac
8        inec_eunac
9        inec_fpnac
10       inec_ftnac
11       inec_glnac
12       inec_gpnac
13       inec_gvnac
14       inec_oenac
15       inec_pcnac
16       inec_sunac
17      inec_vgonac
18      inec_vgpnac
19      inec_vgvnac
20       inec_vmnac
21   rel_inec_acnac
22   rel_inec_adnac
23   rel_inec_apnac
24   rel_inec_cgnac
25   rel_inec_cpnac
26   rel_inec_ctnac
27   rel_inec_eunac
28   rel_inec_fpnac
29   rel_inec_ftnac
30   rel_inec_glnac
31   rel_inec_gpnac
32   rel_inec_gvnac
33   rel_inec_oenac
34   rel_inec_pcnac
35   rel_inec_sunac
36  rel_inec_vgonac
37  rel_inec_vgpnac
38  rel_inec_vgvnac
39   rel_inec_vmnac
40  table_inventory
   table_name  row_count  column_count  has_identificador
0  inec_sunac      90247            29                  1
1  inec_vmnac      56760            23      

## 7) Example analysis query


In [8]:
conn = sqlite3.connect(DB_PATH)
example_sql = "SELECT e.provincia, COUNT(DISTINCT e.identificador) AS n_upas FROM encuestas e GROUP BY e.provincia ORDER BY n_upas DESC LIMIT 15;"
pd.read_sql_query(example_sql, conn)


,provincia,n_upas
0,MANABÍ,4005
1,COTOPAXI,3963
2,CHIMBORAZO,3569
3,TUNGURAHUA,3407
4,GUAYAS,2901
5,AZUAY,2778
6,PICHINCHA,2629
7,LOS RÍOS,2169
8,BOLÍVAR,1832
9,LOJA,1783
